# **Taller: Árboles de decisión**

### **Introducción: Predicción de cáncer de mama**
El objetivo de este taller es construir árboles de decisión que permitan predecir si un paciente tiene cáncer de mama, o no. Además, se busca calcular la exactitud predictiva de los modelos obtenidos y ajustar los hiperparámetros de tal manera que se pueda determinar el conjunto de parámetros que logran una mayor exactitud.

### **Los datos:**
Para la experimentación se usará un conjunto de datos de 569
personas que hicieron parte de un estudio enfocado en el diagnóstico de cáncer de mama. Cada persona se describe utilizando 17 atributos que se obtienen de una imagen digitalizada de una biopsia por aspiración con aguja fina (FNA) de una masa mamaria. Dichos atributos incluyen características de los núcleos celulares presentes en la imagen tales como área, perímetro, radio, textura, entre otra información. La etiqueta de clase o variable dependiente es el atributo
*diagnosis* cuyos valores pueden ser 1 ó 0, donde 1 significa que la persona tiene cáncer de mama y 0 en caso contrario. En este taller se obtendrán modelos que intentan predecir la variable dependiente
*diagnosis*.

In [1]:
import sklearn
import numpy as np
import pandas as pd

### **Manejo de datos:**
Cargando los datos:
1. Leer el archivo ```breastCancer.csv```. Utilice la función info() para verificar si hay, o no, valores nulos en el dataset. Además, utilice la función head() para imprimir los 5 primeros registros.

In [2]:

file_path = "breastCancer.csv"
df = pd.read_csv(file_path)

print("Información del DataFrame:")
df.info()

print("\nPrimeros 5 registros del DataFrame:")
df.head()

Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   radius_mean              569 non-null    float64
 1   texture_mean             569 non-null    float64
 2   perimeter_mean           569 non-null    float64
 3   area_mean                569 non-null    float64
 4   smoothness_mean          569 non-null    float64
 5   compactness_mean         569 non-null    float64
 6   symmetry_mean            569 non-null    float64
 7   fractal_dimension_mean   569 non-null    float64
 8   radius_worst             569 non-null    float64
 9   texture_worst            569 non-null    float64
 10  perimeter_worst          569 non-null    float64
 11  area_worst               569 non-null    float64
 12  smoothness_worst         569 non-null    float64
 13  compactness_worst        569 non-null    float64
 14 

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,symmetry_mean,fractal_dimension_mean,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,symmetry_worst,fractal_dimension_worst,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.2419,0.07871,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.4601,0.11890,1
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.1812,0.05667,24.99,23.41,158.80,1956.0,0.1238,0.1866,275.0000,0.08902,1
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.2069,0.05999,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.3613,0.08758,1
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2597,0.09744,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6638,173.00000,1
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1809,0.05883,22.54,16.67,152.20,1575.0,0.1374,205.0000,0.2364,0.07678,1


2. Seleccionar aleatoriamente el 80% del conjunto de datos para entrenar y el 20% restante para las pruebas.

In [3]:
from sklearn.model_selection import train_test_split, cross_val_score

X = df.drop("diagnosis", axis=1, errors="ignore")
y = df["diagnosis"] #variable objetivo

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Conjunto de entrenamiento: {len(X_train)} registros (80%)")
print(f"Conjunto de prueba: {len(X_test)} registros (20%)")
print("\nForma de los conjuntos de datos:")
print(f" - X_train: {X_train.shape}")
print(f" - X_test: {X_test.shape}")
print(f" - y_train: {y_train.shape}")
print(f" - y_test: {y_test.shape}")

Conjunto de entrenamiento: 455 registros (80%)
Conjunto de prueba: 114 registros (20%)

Forma de los conjuntos de datos:
 - X_train: (455, 16)
 - X_test: (114, 16)
 - y_train: (455,)
 - y_test: (114,)


3. Utilice un pipeline que le permita normalizar los datos. Obtenga los subconjuntos X_train y y_train.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Crear el pipeline de preprocesamiento
preprocessor = Pipeline([
    #("imputer", SimpleImputer(strategy="median")), no es necesario no se vio datos nulos
    ("scaler", StandardScaler())
])

# Ajustar y transformar los datos de entrenamiento
X_train_processed = preprocessor.fit_transform(X_train)

# Convertir a DataFrame para una mejor visualización
X_train_processed_df = pd.DataFrame(X_train_processed, columns=X_train.columns)

print("Pipeline de normalización aplicado a X_train. Primeros 5 registros de X_train normalizado:")
print(X_train_processed_df.head())

print("\nPrimeros 5 registros de y_train (sin cambios):")
print(y_train.head())

Pipeline de normalización aplicado a X_train. Primeros 5 registros de X_train normalizado:
   radius_mean  texture_mean  perimeter_mean  area_mean  smoothness_mean  \
0    -1.440753     -0.435319       -1.362085  -1.139118        -0.213163   
1     1.974096      1.733026        2.091672   1.851973        -0.212826   
2    -1.399982     -1.249622       -1.345209  -1.109785        -0.214481   
3    -0.981797      1.416222       -0.982587  -0.866944        -0.213613   
4    -1.117700     -1.010259       -1.125002  -0.965942        -0.212858   

   compactness_mean  symmetry_mean  fractal_dimension_mean  radius_worst  \
0         -0.165730      -0.295868               -0.122169     -1.232861   
1         -0.159779      -0.295321               -0.122987      2.173314   
2         -0.167986      -0.295425               -0.123536     -1.295284   
3         -0.168622      -0.296280               -0.124336     -0.829197   
4         -0.168275      -0.294651               -0.123540     -1.085129

### **Creación de árboles de decisión**
4. Configurar los hiperparámetros del árbol de decisión de la siguiente manera: criterion=gini, splitter=best, y random_state=123. Obtener 10 árboles de decisión que resultan de modificar el hiperparámetro max_depth desde 1 a 10 con incrementos de 1.

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

results = []
best = []

# Entrenar modelos y calcular accuracy
for i in range(1, 11):
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            criterion="gini",
            splitter="best",
            random_state=123,
            max_depth=i
        ))
    ])
    model_pipeline.fit(X_train, y_train)
    scores = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring='accuracy')
    y_pred = model_pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results.append({
        "criterion": "gini",
        "splitter":"best",
        "max_depth": i, 
        "accuracy": accuracy, 
        "mean_cv_accuracy": scores.mean(), 
        "std_cv_accuracy": scores.std()
    })


5. Incluya en el notebook una tabla con el accuracy para los 10 árboles del punto anterior. Los valores del accuracy se calculan sobre el conjunto de prueba, es decir, sobre las predicciones que hace el modelo para las instancias que se encuentran en X_test

In [6]:
# Mostrar resultados en una tabla
print("Tabla de Accuracy para árboles con criterion='gini' y splitter='best':")
results_df = pd.DataFrame(results)

print("Tabla de Accuracy promedio con Validación Cruzada (ordenada de mejor a peor):")
results_df_sorted = results_df.sort_values(by='mean_cv_accuracy', ascending=False)
print(results_df_sorted.to_string())

# Identificamos los mejores hiperparámetros
best_params = results_df_sorted.iloc[0]
best.append(best_params)
print("\n--- Mejor Combinación de Hiperparámetros ---")
print(f"Criterion: gini")
print(f"Splitter: best")
#print(f"Accuracy: {best_params['accuracy']:.4f}")
print(f"Max Depth: {int(best_params['max_depth'])}")
print(f"Accuracy promedio en CV: {best_params['mean_cv_accuracy']:.4f}")

Tabla de Accuracy para árboles con criterion='gini' y splitter='best':
Tabla de Accuracy promedio con Validación Cruzada (ordenada de mejor a peor):
  criterion splitter  max_depth  accuracy  mean_cv_accuracy  std_cv_accuracy
4      gini     best          5  0.956140          0.929670         0.025631
3      gini     best          4  0.956140          0.925275         0.022413
5      gini     best          6  0.956140          0.914286         0.018906
6      gini     best          7  0.956140          0.914286         0.025441
7      gini     best          8  0.956140          0.914286         0.025441
8      gini     best          9  0.956140          0.914286         0.025441
9      gini     best         10  0.956140          0.914286         0.025441
2      gini     best          3  0.982456          0.903297         0.022413
0      gini     best          1  0.947368          0.892308         0.035710
1      gini     best          2  0.929825          0.887912         0.023466

---

6. Utilice ahora los hiperparámetros criterion=entropy, splitter=best, y random_state=123. Obtenga 10 árboles de decisión que resultan de modificar el hiperparámetro max_depth desde 1 a 10 con incrementos de 1. Calcule el accuracy sobre el conjunto de prueba. Incluya en el notebook una tabla con los 10 valores del accuracy.

In [7]:
results = []
# Entrenar modelos con criterion='entropy' y calcular accuracy
for i in range(1, 11):
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            criterion="entropy",
            splitter="best",
            random_state=123,
            max_depth=i
        ))
    ])
    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    scores = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring='accuracy')

    results.append({
        "criterion": "entropy",
        "splitter":"best",
        "max_depth": i, 
        "accuracy": accuracy, 
        "mean_cv_accuracy": scores.mean(), 
        "std_cv_accuracy": scores.std()
    })


In [8]:
# Mostrar resultados en una tabla
print("Tabla de Accuracy para árboles con criterion='entropy' y splitter='best':")
results_df = pd.DataFrame(results)

print("Tabla de Accuracy promedio con Validación Cruzada (ordenada de mejor a peor):")
results_df_sorted = results_df.sort_values(by='mean_cv_accuracy', ascending=False)
print(results_df_sorted.to_string())

# Identificamos los mejores hiperparámetros
best_params = results_df_sorted.iloc[0]
best.append(best_params)
print("\n--- Mejor Combinación de Hiperparámetros ---")
print(f"Criterion: entropy")
print(f"Splitter: best")
#print(f"Accuracy: {best_params['accuracy']:.4f}")
print(f"Max Depth: {int(best_params['max_depth'])}")
print(f"Accuracy promedio en CV: {best_params['mean_cv_accuracy']:.4f}")

Tabla de Accuracy para árboles con criterion='entropy' y splitter='best':
Tabla de Accuracy promedio con Validación Cruzada (ordenada de mejor a peor):
  criterion splitter  max_depth  accuracy  mean_cv_accuracy  std_cv_accuracy
2   entropy     best          3  0.938596          0.925275         0.036380
4   entropy     best          5  0.947368          0.923077         0.030295
5   entropy     best          6  0.947368          0.920879         0.032151
3   entropy     best          4  0.938596          0.920879         0.027274
6   entropy     best          7  0.947368          0.914286         0.033620
7   entropy     best          8  0.947368          0.914286         0.033620
8   entropy     best          9  0.947368          0.914286         0.033620
9   entropy     best         10  0.947368          0.914286         0.033620
1   entropy     best          2  0.938596          0.890110         0.023051
0   entropy     best          1  0.938596          0.881319         0.028991



7. Utilice los hiperparámetros criterion=entropy, splitter=random, y random_state=123. Obtenga 10 árboles de decisión que resultan de modificar el hiperparámetro max_depth desde 1 a 10 con incrementos de 1. Calcule el accuracy sobre el conjunto de prueba. Incluya en el notebook una tabla con los 10 valores del accuracy.

In [9]:
results = []
# Entrenar modelos con criterion='entropy', splitter='random' y calcular accuracy
for i in range(1, 11):
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            criterion="entropy",
            splitter="random",
            random_state=123,
            max_depth=i
        ))
    ])
    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    scores = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring='accuracy')

    results.append({
        "criterion": "entropy",
        "splitter":"random",
        "max_depth": i, 
        "accuracy": accuracy, 
        "mean_cv_accuracy": scores.mean(), 
        "std_cv_accuracy": scores.std()
    })


In [10]:
# Mostrar resultados en una tabla
print("Tabla de Accuracy para árboles con criterion='entropy' y splitter='random':")
results_df = pd.DataFrame(results)

print("Tabla de Accuracy promedio con Validación Cruzada (ordenada de mejor a peor):")
results_df_sorted = results_df.sort_values(by='mean_cv_accuracy', ascending=False)
print(results_df_sorted.to_string())

# Identificamos los mejores hiperparámetros
best_params = results_df_sorted.iloc[0]
best.append(best_params)
print("\n--- Mejor Combinación de Hiperparámetros ---")
print(f"Criterion: entropy")
print(f"Splitter: random")
#print(f"Accuracy: {best_params['accuracy']:.4f}")
print(f"Max Depth: {int(best_params['max_depth'])}")
print(f"Accuracy promedio en CV: {best_params['mean_cv_accuracy']:.4f}")

Tabla de Accuracy para árboles con criterion='entropy' y splitter='random':
Tabla de Accuracy promedio con Validación Cruzada (ordenada de mejor a peor):
  criterion splitter  max_depth  accuracy  mean_cv_accuracy  std_cv_accuracy
5   entropy   random          6  0.964912          0.936264         0.024474
6   entropy   random          7  0.973684          0.936264         0.029812
4   entropy   random          5  0.929825          0.934066         0.013900
7   entropy   random          8  0.973684          0.931868         0.022413
9   entropy   random         10  0.921053          0.929670         0.037812
3   entropy   random          4  0.921053          0.918681         0.037168
2   entropy   random          3  0.921053          0.914286         0.027274
8   entropy   random          9  0.921053          0.912088         0.019658
0   entropy   random          1  0.938596          0.909890         0.031391
1   entropy   random          2  0.938596          0.903297         0.037684

8. Indique los hiperparámetros que por el momento le permiten obtener el árbol con mayor accuracy

In [11]:
df_best = pd.DataFrame(best)

# Ordenar por 'mean_cv_accuracy' de mayor a menor
df_best_sorted = df_best.sort_values(by='mean_cv_accuracy', ascending=False)

print(df_best_sorted)
best_params = df_best_sorted.iloc[0]
best_params

  criterion splitter  max_depth  accuracy  mean_cv_accuracy  std_cv_accuracy
5   entropy   random          6  0.964912          0.936264         0.024474
4      gini     best          5  0.956140          0.929670         0.025631
2   entropy     best          3  0.938596          0.925275         0.036380


criterion            entropy
splitter              random
max_depth                  6
accuracy            0.964912
mean_cv_accuracy    0.936264
std_cv_accuracy     0.024474
Name: 5, dtype: object

9. Seleccione uno de los hiperparámetros disponibles en la documentación (https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) que sea diferente al criterion, splitter, max_depth, y random_state. Indicar el hiperparámetro seleccionado.


**`min_samples_leaf`**
Este parámetro define cuántas muestras como mínimo necesita un nodo para convertirse en una "hoja" (es decir, un nodo final sin más divisiones). Sirve sobre todo para evitar que el modelo se sobreajuste.

* Si pones un valor bajo, el árbol puede crear hojas muy específicas para pocos datos, lo que hace que prácticamente memorice el entrenamiento.
* Si usas un valor más alto, el árbol se ve obligado a ser más general, lo que suele ayudar a que funcione mejor con datos nuevos.

10. Realice dos variaciones en el hiperparámetro seleccionado manteniendo los otros hiperparámetros del punto 8. Indique el accuracy obtenido al modificar el hiperparámetro seleccionado y analice si el árbol de decisión mejora, empeora, o mantiene su exactitud.

In [18]:
# Hiperparámetros del mejor modelo (punto 8)
base_hyperparameters = {
    "criterion": best_params["criterion"],
    "splitter":  best_params["splitter"],
    "max_depth": best_params["max_depth"],
    "random_state": 123
}

# Variaciones para min_samples_leaf
best_leaf_value = 0
min_samples_leaf_variations = [5, 10]
original_mean_cv_accuracy = best_params["mean_cv_accuracy"]

print(f"Analizando el efecto de 'min_samples_leaf' sobre el mejor modelo (Accuracy Prmedio original = {original_mean_cv_accuracy:.6f})\n" + "-"*80)

for leaf_value in min_samples_leaf_variations:
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(**base_hyperparameters, min_samples_leaf=leaf_value))
    ])
    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)
    scores = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring='accuracy')
    mean_cv_accuracy =scores.mean()

    print(f"Para min_samples_leaf = {leaf_value}:")
    print(f"  - Accuracy obtenido: {accuracy:.6f}")

    if mean_cv_accuracy > original_mean_cv_accuracy:
        print("  - Análisis: La exactitud del modelo MEJORÓ.")
        best_leaf_value = leaf_value
    elif mean_cv_accuracy < original_mean_cv_accuracy:
        print("  - Análisis: La exactitud del modelo EMPEORÓ.")
    else:
        print("  - Análisis: La exactitud del modelo SE MANTUVO IGUAL.")
    print("-"*80)

Analizando el efecto de 'min_samples_leaf' sobre el mejor modelo (Accuracy Prmedio original = 0.936264)
--------------------------------------------------------------------------------
Para min_samples_leaf = 5:
  - Accuracy obtenido: 0.921053
  - Análisis: La exactitud del modelo MEJORÓ.
--------------------------------------------------------------------------------
Para min_samples_leaf = 10:
  - Accuracy obtenido: 0.921053
  - Análisis: La exactitud del modelo EMPEORÓ.
--------------------------------------------------------------------------------


11. Calcule y muestre la importancia de las variables del modelo obtenido en el punto anterior usando la característica feature_importances. Analice los resultados e indique en el notebook cuáles son las variables más relevantes en este problema.

In [19]:
print(best_params)
# Entrenar el mejor modelo para obtener las importancias
best_model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        criterion=best_params["criterion"],
        splitter=best_params["splitter"],
        max_depth=best_params["max_depth"],
        random_state=123,
        min_samples_leaf=best_leaf_value
    ))
])
best_model_pipeline.fit(X_train, y_train)

# Extraer el clasificador y las importancias
feature_importances = best_model_pipeline.named_steps['classifier'].feature_importances_
feature_names = X_train.columns

# Crear un DataFrame para visualizar
importances_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances
}).sort_values(by='importance', ascending=False)

print("Importancia de las variables para el mejor modelo:")
print(importances_df)

criterion            entropy
splitter              random
max_depth                  6
accuracy            0.964912
mean_cv_accuracy    0.936264
std_cv_accuracy     0.024474
Name: 5, dtype: object
Importancia de las variables para el mejor modelo:
                    feature  importance
10          perimeter_worst    0.801255
1              texture_mean    0.066877
9             texture_worst    0.051215
2            perimeter_mean    0.023242
0               radius_mean    0.021746
6             symmetry_mean    0.015381
7    fractal_dimension_mean    0.015113
11               area_worst    0.005170
3                 area_mean    0.000000
4           smoothness_mean    0.000000
5          compactness_mean    0.000000
8              radius_worst    0.000000
12         smoothness_worst    0.000000
13        compactness_worst    0.000000
14           symmetry_worst    0.000000
15  fractal_dimension_worst    0.000000
